In [3]:
import pandas as pd
import unicodedata
import re
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path

In [4]:
#Visualización de las primeras filas 

DATA_PATH = Path('/Users/valemoravale/Documents/UNIVERSIDAD /Semestre 5/ETL/workshop3/data/2015.csv')
df = pd.read_csv(DATA_PATH)
df.head()

,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176


In [5]:
#Numero de filas, columnas y tipo de cada una
print('Rows:', len(df))
print('Columns:', df.shape[1])
df.info()

Rows: 158
Columns: 12
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        158 non-null    object 
 1   Region                         158 non-null    object 
 2   Happiness Rank                 158 non-null    int64  
 3   Happiness Score                158 non-null    float64
 4   Standard Error                 158 non-null    float64
 5   Economy (GDP per Capita)       158 non-null    float64
 6   Family                         158 non-null    float64
 7   Health (Life Expectancy)       158 non-null    float64
 8   Freedom                        158 non-null    float64
 9   Trust (Government Corruption)  158 non-null    float64
 10  Generosity                     158 non-null    float64
 11  Dystopia Residual              158 non-null    float64
dtypes: float64(9), int64(1), obj

In [6]:
#Valores faltantes y su porcentaje 
missing = df.isna().sum().to_frame('missing_count')
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
missing

,missing_count,missing_percent
Country,0,0.0
Region,0,0.0
Happiness Rank,0,0.0
Happiness Score,0,0.0
Standard Error,0,0.0
Economy (GDP per Capita),0,0.0
Family,0,0.0
Health (Life Expectancy),0,0.0
Freedom,0,0.0
Trust (Government Corruption),0,0.0


In [7]:
#Columnas duplicadas y Country duplicado 
print('Full duplicated rows:', df.duplicated().sum())
print('Duplicated Country:', df['Country'].duplicated().sum())

Full duplicated rows: 0
Duplicated Country: 0


In [8]:
# Crear función para limpiar nombres
def limpiar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto

# Crear columna limpia
df["country_clean"] = df["Country"].apply(limpiar_texto)

# Buscar duplicados exactos después de limpiar
duplicados_limpios = df[df.duplicated("country_clean", keep=False)]

print("Duplicados después de limpiar:")
print(duplicados_limpios[["Country", "country_clean"]])

Duplicados después de limpiar:
Empty DataFrame
Columns: [Country, country_clean]
Index: []


In [9]:
countries = df["Country"].dropna().unique()

posibles_duplicados = []

for pais1, pais2 in combinations(countries, 2):
    pais1_clean = limpiar_texto(pais1)
    pais2_clean = limpiar_texto(pais2)

    similitud = SequenceMatcher(None, pais1_clean, pais2_clean).ratio()

    if similitud >= 0.85:
        posibles_duplicados.append({
            "Country 1": pais1,
            "Country 2": pais2,
            "Similitud": round(similitud, 3)
        })

posibles_duplicados_df = pd.DataFrame(posibles_duplicados)

print(posibles_duplicados_df)

   Country 1 Country 2  Similitud
0    Iceland   Ireland      0.857
1  Australia   Austria      0.875


In [10]:
def limpiar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto

# Crear columna limpia para Region
df["region_clean"] = df["Region"].apply(limpiar_texto)

# Ver regiones originales y limpias
print(df[["Region", "region_clean"]].drop_duplicates())

                             Region                     region_clean
0                    Western Europe                   western europe
4                     North America                    north america
8         Australia and New Zealand        australia and new zealand
10  Middle East and Northern Africa  middle east and northern africa
11      Latin America and Caribbean      latin america and caribbean
23                Southeastern Asia                southeastern asia
30       Central and Eastern Europe       central and eastern europe
37                     Eastern Asia                     eastern asia
70               Sub-Saharan Africa                subsaharan africa
78                    Southern Asia                    southern asia


In [11]:
regions = df["Region"].dropna().unique()

posibles_duplicados_region = []

for region1, region2 in combinations(regions, 2):
    region1_clean = limpiar_texto(region1)
    region2_clean = limpiar_texto(region2)

    similitud = SequenceMatcher(None, region1_clean, region2_clean).ratio()

    if similitud >= 0.85:
        posibles_duplicados_region.append({
            "Region 1": region1,
            "Region 2": region2,
            "Similitud": round(similitud, 3)
        })

posibles_duplicados_region_df = pd.DataFrame(posibles_duplicados_region)

print(posibles_duplicados_region_df)

            Region 1       Region 2  Similitud
0  Southeastern Asia  Southern Asia      0.867


In [19]:
# Seleccionar solo columnas numéricas
numeric_cols = df.select_dtypes(include=["number"]).columns

ceros = (df[numeric_cols] == 0).sum()
negativos = (df[numeric_cols] < 0).sum()

print("Ceros por columna:")
print(ceros)
print("-------------------------------------")
print("Negativos por columna:")
print(negativos)

Ceros por columna:
Happiness Rank                   0
Happiness Score                  0
Standard Error                   0
Economy (GDP per Capita)         1
Family                           1
Health (Life Expectancy)         1
Freedom                          1
Trust (Government Corruption)    1
Generosity                       1
Dystopia Residual                0
dtype: int64
-------------------------------------
Negativos por columna:
Happiness Rank                   0
Happiness Score                  0
Standard Error                   0
Economy (GDP per Capita)         0
Family                           0
Health (Life Expectancy)         0
Freedom                          0
Trust (Government Corruption)    0
Generosity                       0
Dystopia Residual                0
dtype: int64


In [13]:
filas_no_validas = df[df[numeric_cols].le(0).any(axis=1)]

print(filas_no_validas)

                      Country                           Region  \
73                  Indonesia                Southeastern Asia   
101                    Greece                   Western Europe   
111                      Iraq  Middle East and Northern Africa   
119          Congo (Kinshasa)               Sub-Saharan Africa   
122              Sierra Leone               Sub-Saharan Africa   
147  Central African Republic               Sub-Saharan Africa   

     Happiness Rank  Happiness Score  Standard Error  \
73               74            5.399         0.02596   
101             102            4.857         0.05062   
111             112            4.677         0.05232   
119             120            4.517         0.03680   
122             123            4.507         0.07068   
147             148            3.678         0.06112   

     Economy (GDP per Capita)   Family  Health (Life Expectancy)  Freedom  \
73                    0.82827  1.08708                   0.63793  0

In [14]:
#ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport (df, title="Raw sale data profiling report", explorative= True)
profile.to_file("raw_sale_data_data_profile.html2015" )

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  7.37it/s]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ydata_profiling/profile_report.py:386: UserWarning: Extension .html2015 not supported. For now we assume .html was intended. To remove this warning, please use .html or .json.
  warnings.warn(
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 254.60it/s]
